# DDPM Reproduction & Inference Demo
**Course:** CS-4112 Deep Learning — FAST-NUCES Islamabad  
**Project:** Denoising Diffusion Probabilistic Models (DDPM) on CIFAR-10

This notebook provides a runnable interface to generate images using the models reproduced in this project. It supports both the standard **DDPM** (1000 steps) and the accelerated **DDIM** sampler.

### 1. Setup & Installation
Install the necessary libraries to run inference.

In [ ]:
!pip install -q diffusers accelerate torch torchvision matplotlib

### 2. Imports

In [ ]:
import torch
from diffusers import DDPMPipeline, DDIMScheduler
import matplotlib.pyplot as plt
import numpy as np

### 3. Load Pre-trained Model
We use the official `google/ddpm-cifar10-32` checkpoint, which we validated in Assignment 2.

In [ ]:
model_id = "google/ddpm-cifar10-32"
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading model to {device}...")
ddpm = DDPMPipeline.from_pretrained(model_id).to(device)
print("Model loaded successfully!")

### 4. Standard DDPM Inference (1000 Steps)
Generate a batch of CIFAR-10 images using the original stochastic sampling.

In [ ]:
num_images = 8
print(f"Generating {num_images} images with DDPM (1000 steps)...")

with torch.no_grad():
    images = ddpm(batch_size=num_images).images

# Visualization
fig, axes = plt.subplots(1, num_images, figsize=(15, 3))
for i, img in enumerate(images):
    axes[i].imshow(img)
    axes[i].axis("off")
plt.suptitle("Generated CIFAR-10 Samples (DDPM - 1000 Steps)")
plt.show()

### 5. Accelerated Inference with DDIM (Assignment 3 Extension)
As explored in Experiment 1 and 2, we can use the **DDIM** scheduler to generate high-quality images in significantly fewer steps (e.g., 50 steps instead of 1000).

In [ ]:
# Switch to DDIM scheduler
ddpm.scheduler = DDIMScheduler.from_config(ddpm.scheduler.config)

inference_steps = 50
eta = 0.0 # Deterministic sampling as per Exp 2 findings

print(f"Generating {num_images} images with DDIM ({inference_steps} steps, eta={eta})...")
with torch.no_grad():
    images_ddim = ddpm(batch_size=num_images, num_inference_steps=inference_steps, eta=eta).images

# Visualization
fig, axes = plt.subplots(1, num_images, figsize=(15, 3))
for i, img in enumerate(images_ddim):
    axes[i].imshow(img)
    axes[i].axis("off")
plt.suptitle(f"Generated CIFAR-10 Samples (DDIM - {inference_steps} Steps)")
plt.show()

### 6. Conclusion
This notebook demonstrates the correct functioning of the U-Net architecture and the noise prediction pipeline. For detailed quantitative results (FID), please refer to the `README.md` and the final project report.